## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *
from resnet_20_32_44_56_v2 import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

# net = ResNet20()
# net = ResNet32()
net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 273ms | Tot: 32s63ms | Loss: 2.024 | Acc: 26.302% (13151/50000) 391/391 
  Step: 17ms | Tot: 1s789ms | Loss: 1.923 | Acc: 32.960% (3296/10000) 100/100 100 
Saving..

Epoch: 1
  Step: 80ms | Tot: 31s940ms | Loss: 1.500 | Acc: 44.598% (22299/50000) 391/391 
  Step: 16ms | Tot: 1s735ms | Loss: 1.277 | Acc: 54.420% (5442/10000) 100/100 
Saving..

Epoch: 2
  Step: 80ms | Tot: 31s691ms | Loss: 1.156 | Acc: 58.634% (29317/50000) 391/391 
  Step: 19ms | Tot: 1s748ms | Loss: 1.141 | Acc: 60.390% (6039/10000) 100/100 
Saving..

Epoch: 3
  Step: 86ms | Tot: 31s937ms | Loss: 0.959 | Acc: 66.012% (33006/50000) 391/391 
  Step: 17ms | Tot: 1s804ms | Loss: 1.057 | Acc: 64.010% (6401/10000) 100/100 
Saving..

Epoch: 4
  Step: 83ms | Tot: 32s180ms | Loss: 0.820 | Acc: 71.150% (35575/50000) 391/391 
  Step: 17ms | Tot: 1s779ms | Loss: 0.925 | Acc: 69.160% (6916/10000) 100/100 
Saving..

Epoch: 5
  Step: 82ms | Tot: 32s290ms | Loss: 0.726 | Acc: 74.578% (37289/50000) 391/391 
  Step: 17

  Step: 82ms | Tot: 32s71ms | Loss: 0.240 | Acc: 91.682% (45841/50000) 391/391  
  Step: 17ms | Tot: 1s778ms | Loss: 0.473 | Acc: 85.920% (8592/10000) 100/100 100 80/100 

Epoch: 48
  Step: 83ms | Tot: 32s283ms | Loss: 0.238 | Acc: 91.566% (45783/50000) 391/391 
  Step: 18ms | Tot: 1s791ms | Loss: 0.424 | Acc: 86.940% (8694/10000) 100/100  16/100 
Saving..

Epoch: 49
  Step: 83ms | Tot: 32s346ms | Loss: 0.237 | Acc: 91.686% (45843/50000) 391/391 
  Step: 17ms | Tot: 1s762ms | Loss: 0.485 | Acc: 84.900% (8490/10000) 100/100 

Epoch: 50
  Step: 86ms | Tot: 32s84ms | Loss: 0.235 | Acc: 91.714% (45857/50000) 391/391  
  Step: 17ms | Tot: 1s778ms | Loss: 0.508 | Acc: 84.320% (8432/10000) 100/100 /100 

Epoch: 51
  Step: 85ms | Tot: 32s165ms | Loss: 0.239 | Acc: 91.612% (45806/50000) 391/391 
  Step: 16ms | Tot: 1s796ms | Loss: 0.495 | Acc: 85.130% (8513/10000) 100/100 00 26/100 45/100 

Epoch: 52
  Step: 83ms | Tot: 32s161ms | Loss: 0.230 | Acc: 91.974% (45987/50000) 391/391 
  Step: 17ms |

  Step: 17ms | Tot: 1s756ms | Loss: 0.381 | Acc: 91.690% (9169/10000) 100/100 

Epoch: 140
  Step: 83ms | Tot: 32s303ms | Loss: 0.014 | Acc: 99.596% (49798/50000) 391/391 
  Step: 17ms | Tot: 1s783ms | Loss: 0.381 | Acc: 91.660% (9166/10000) 100/100 

Epoch: 141
  Step: 83ms | Tot: 32s364ms | Loss: 0.014 | Acc: 99.636% (49818/50000) 391/391 
  Step: 17ms | Tot: 1s776ms | Loss: 0.383 | Acc: 91.620% (9162/10000) 100/100 

Epoch: 142
  Step: 83ms | Tot: 32s381ms | Loss: 0.015 | Acc: 99.596% (49798/50000) 391/391 
  Step: 18ms | Tot: 1s770ms | Loss: 0.378 | Acc: 92.030% (9203/10000) 100/100 
Saving..

Epoch: 143
  Step: 85ms | Tot: 32s461ms | Loss: 0.013 | Acc: 99.654% (49827/50000) 391/391 
  Step: 17ms | Tot: 1s780ms | Loss: 0.377 | Acc: 91.750% (9175/10000) 100/100 00 

Epoch: 144
  Step: 81ms | Tot: 32s483ms | Loss: 0.014 | Acc: 99.614% (49807/50000) 391/391 
  Step: 17ms | Tot: 1s792ms | Loss: 0.388 | Acc: 91.850% (9185/10000) 100/100 /100 

Epoch: 145
  Step: 81ms | Tot: 32s287ms | L

In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 92.04
Error: 7.96
